# my3d on Google Colab (free GPU)

Image in → 3D mesh (GLB) out.

**First:** *Runtime → Change runtime type → T4 GPU*, then run the cells top to bottom.
The first run takes about 10 minutes (install + model download).

In [ ]:
# 1. Check the GPU (should show a Tesla T4 or better)
!nvidia-smi

In [ ]:
# 2. Get the code
%cd /content
!git clone -q https://github.com/raj-anand-14/my3d
%cd /content/my3d
!git clone -q --depth 1 https://github.com/VAST-AI-Research/TripoSG

In [ ]:
# 3. Install (Colab already has PyTorch + CUDA; ~3-5 min)
# Lock Colab's own GPU build of PyTorch so nothing below can replace it
!pip freeze | grep -E '^(torch|torchvision|torchaudio)==' > /tmp/constraints.txt; cat /tmp/constraints.txt
!grep -v -E '^(diso|numpy)' TripoSG/requirements.txt > /tmp/req.txt
!pip install -q -c /tmp/constraints.txt -r /tmp/req.txt -r requirements-app.txt
# Upgrade to the library versions my3d was tested with (Colab ships older ones)
!pip install -q -c /tmp/constraints.txt -U diffusers==0.41.0 transformers==5.20.2 peft==0.21.3 huggingface_hub==1.33.0
# Colab's preinstalled torchao is too old for new diffusers and isn't needed: remove it
!pip uninstall -y -q torchao
# Optional speed-up: diso (fast GPU mesh extraction). If its build fails, that's OK:
# my3d automatically falls back to scikit-image extraction.
!pip install -q -c /tmp/constraints.txt --no-build-isolation diso || echo 'diso not installed: using the fallback, which is fine'
# Sanity check (fresh process, so it sees the installed versions)
!python -c "import torch,diffusers,transformers;print('torch',torch.__version__,'| CUDA OK:',torch.cuda.is_available(),'| diffusers',diffusers.__version__,'| transformers',transformers.__version__)"

In [ ]:
# 4. Start the my3d server in the background and wait for the models to load
import subprocess, time
log = open('/content/my3d/server.log', 'w')
proc = subprocess.Popen(['python', 'server.py'], cwd='/content/my3d', stdout=log, stderr=subprocess.STDOUT)
for _ in range(180):
    time.sleep(5)
    txt = open('/content/my3d/server.log').read()
    if 'Uvicorn running' in txt or 'Application startup complete' in txt:
        print('Server is up!'); break
    if proc.poll() is not None:
        print(txt[-6000:]); raise RuntimeError('Server crashed, see log above')
else:
    print(open('/content/my3d/server.log').read()[-3000:])

In [ ]:
# 5. Open the web app (opens in a new tab: upload an image, get a 3D model)
from google.colab.output import serve_kernel_port_as_window
serve_kernel_port_as_window(8000)

### Optional: generate without the web page
Upload an image, get a `.glb` downloaded straight to your computer.

In [ ]:
import requests
from google.colab import files
up = files.upload()
name = next(iter(up))
r = requests.post('http://localhost:8000/generate', files={'image': (name, up[name])}, data={'faces': 20000}, timeout=600)
r.raise_for_status()
out = name.rsplit('.', 1)[0] + '.glb'
open(out, 'wb').write(r.content)
files.download(out)

**If something fails:** run `!tail -50 /content/my3d/server.log` and share the output.

In [ ]:
# Debug: show the end of the server log
!tail -60 /content/my3d/server.log